# 17. Hierarchical support interventions and blocked factorial inference

![Hierarchical support and blocked inference](../images/17_hierarchical_support_and_factorial_inference.svg)

**Learning goals:** compute frozen and resampled temporal support, preserve eight paired four-cell model blocks, calculate the interaction and direct allocation contrast, compare raw and clipped-logit scales, apply materiality and equivalence rules, resample blocks and participants safely, and validate a complete protocol registry. All examples are synthetic.

In [ ]:
from itertools import product
import hashlib
import json
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 17
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
print(f"NumPy {np.__version__}; seed={SEED}; synthetic data only")

## 1. Temporal capacity and expected realized support

A 16-frame clip uses anchors spaced by eight frames. The frozen policy exposes one anchor per sequence and replicate. The resampled policy can visit every separated anchor. The occupancy equations below calculate how many sequence-anchor pairs are expected to appear at least once after a fixed number of draws. Capacity, realized support, and statistical independence are different ideas.

In [ ]:
def anchors(frame_count, clip_length=16, spacing=8):
    valid_starts = frame_count - clip_length + 1
    if valid_starts <= 0:
        return np.empty(0, dtype=int)
    return np.arange(0, valid_starts, spacing, dtype=int)

def expected_frozen_support(sequence_count, draws):
    unseen = np.exp(draws * np.log1p(-1.0 / sequence_count))
    return sequence_count * (1.0 - unseen)

def expected_resampled_support(anchor_counts, draws):
    anchor_counts = np.asarray(anchor_counts, dtype=float)
    sequence_count = len(anchor_counts)
    unseen = np.exp(draws * np.log1p(-1.0 / (sequence_count * anchor_counts)))
    return np.sum(anchor_counts * (1.0 - unseen))

example_anchors = anchors(65)
frame_counts = rng.integers(40, 121, size=200)
K = np.array([len(anchors(int(n))) for n in frame_counts])
C = 10_000
E_F = expected_frozen_support(len(K), C)
E_R = expected_resampled_support(K, C)
nonoverlap_K = np.array([len(anchors(int(n), spacing=16)) for n in frame_counts])
assert example_anchors.tolist() == [0, 8, 16, 24, 32, 40, 48]
assert np.all(K >= nonoverlap_K) and E_R > E_F
print(f"median K={np.median(K):.1f}; frozen E={E_F:.1f}; resampled E={E_R:.1f}; ratio={E_R/E_F:.2f}")

## 2. Simulate eight complete four-cell blocks

The score array has axes `(block, sequence support, temporal policy, participant)`. Index 0 is low or frozen, and index 1 is high or resampled. Participant effects are shared across all cells, while block effects represent training variation. The complete array preserves every matched comparison.

In [ ]:
R, P = 8, 308
gfc_centers = np.array([[0.55, 0.68], [0.70, 0.75]])
completion_centers = np.array([[0.50, 0.58], [0.62, 0.66]])
participant_effect = rng.normal(0, 0.045, size=(1, 1, 1, P))
block_effect = rng.normal(0, 0.008, size=(R, 1, 1, 1))
cell_jitter = rng.normal(0, 0.006, size=(R, 2, 2, 1))
gfc = np.clip(gfc_centers[None, :, :, None] + participant_effect + block_effect
              + cell_jitter + rng.normal(0, 0.018, size=(R, 2, 2, P)), 0, 1)
completion = np.clip(completion_centers[None, :, :, None] + 0.8 * participant_effect
                     + 0.5 * block_effect + rng.normal(0, 0.018, size=(R, 2, 2, P)), 0, 1)
Y = gfc.mean(axis=-1, dtype=np.float64)
C_top1 = completion.mean(axis=-1, dtype=np.float64)
assert gfc.shape == completion.shape == (8, 2, 2, 308)
assert Y.shape == (8, 2, 2) and np.all((0 <= Y) & (Y <= 1))
print("mean four-cell GFC-v2 table:\n", Y.mean(axis=0))

## 3. Interaction, simple effects, allocation, and completion gap

The interaction can be calculated from either pair of simple effects. A negative value means temporal resampling helps more at low sequence support. The direct allocation contrast compares low-resampled with high-frozen. The completion-gap interaction repeats the calculation after subtracting independent completion from GFC-v2.

In [ ]:
def factorial_contrasts(cell_values):
    LF, LR = cell_values[..., 0, 0], cell_values[..., 0, 1]
    HF, HR = cell_values[..., 1, 0], cell_values[..., 1, 1]
    T_L, T_H = LR - LF, HR - HF
    S_F, S_R = HF - LF, HR - LR
    interaction = T_H - T_L
    assert np.allclose(interaction, S_R - S_F)
    return {"T_L": T_L, "T_H": T_H, "S_F": S_F, "S_R": S_R,
            "I": interaction, "A": LR - HF}

effects = factorial_contrasts(Y)
gap_effects = factorial_contrasts(Y - C_top1)
for name in ("T_L", "T_H", "S_F", "S_R", "I", "A"):
    print(f"{name}: {effects[name].mean():+.4f}")
print(f"J completion-gap interaction: {gap_effects['I'].mean():+.4f}")
assert effects["I"].shape == (8,) and effects["I"].mean() < 0

## 4. Block-level intervals and exact decision definitions

Student intervals use eight block values. Material positivity requires a 95 percent interval above zero and a point estimate at least as large as the margin. Material negativity is symmetric. No material harm requires the 95 percent lower bound to exceed the negative margin. Equivalence requires the 90 percent interval to fit inside the declared band. For the completion-gap interaction $J$, the frozen margin is $\delta_G=0.0625$. We call $J$ resolved only when its 95 percent interval excludes zero and its point estimate reaches the margin. We call it equivalent only when its 90 percent interval lies inside the margin. Otherwise it remains unresolved.

In [ ]:
def t_interval(values, confidence):
    values = np.asarray(values, dtype=float)
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(len(values))
    critical = stats.t.ppf((1 + confidence) / 2, len(values) - 1)
    return mean, (mean - critical * se, mean + critical * se)

def decision(values, margin):
    mean, ci95 = t_interval(values, 0.95)
    _, ci90 = t_interval(values, 0.90)
    return {
        "mean": mean, "ci95": ci95, "ci90": ci90,
        "material_positive": ci95[0] > 0 and mean >= margin,
        "material_negative": ci95[1] < 0 and mean <= -margin,
        "no_material_harm": ci95[0] > -margin,
        "equivalent": ci90[0] > -margin and ci90[1] < margin,
    }

delta_t = delta_i = delta_a = delta_g = 0.0625
effect_margins = {"T_L": delta_t, "T_H": delta_t, "S_F": delta_t,
                  "S_R": delta_t, "I": delta_i, "A": delta_a}
decisions = {name: decision(values, effect_margins[name]) for name, values in effects.items()}
gap_decision = decision(gap_effects["I"], delta_g)
gap_resolved = gap_decision["material_positive"] or gap_decision["material_negative"]
gap_equivalent = gap_decision["equivalent"]
print("I decision:", decisions["I"])
print("J decision:", gap_decision)
assert not (gap_resolved and gap_equivalent)

## 5. Raw and clipped-logit interactions

An additive interaction is not invariant under a nonlinear transformation. The planned cell pattern has the same negative sign on raw and clipped-logit scales. A separate constructed example shows a sign reversal: a smaller raw gain near the ceiling can be a larger gain in log odds. Such a reversal blocks the substitution label.

In [ ]:
epsilon = 1.0 / (2 * 308 * 16)

def clipped_logit(values):
    p = np.clip(values, epsilon, 1 - epsilon)
    return np.log(p) - np.log1p(-p)

logit_effects = factorial_contrasts(clipped_logit(Y))
scale_stable = np.sign(effects["I"].mean()) == np.sign(logit_effects["I"].mean())
reversal_cells = np.array([[0.40, 0.50], [0.90, 0.95]])
reversal_raw = factorial_contrasts(reversal_cells)["I"]
reversal_logit = factorial_contrasts(clipped_logit(reversal_cells))["I"]
scale_stable_negative = effects["I"].mean() < 0 and logit_effects["I"].mean() < 0
substitution = (decisions["T_L"]["material_positive"]
                and decisions["S_F"]["material_positive"]
                and decisions["T_H"]["no_material_harm"]
                and decisions["S_R"]["no_material_harm"]
                and decisions["I"]["material_negative"]
                and scale_stable_negative)
full_replacement = substitution and decisions["A"]["equivalent"]
assert scale_stable and reversal_raw < 0 < reversal_logit
assert not (reversal_raw < 0 and reversal_logit < 0)
assert substitution and full_replacement
print(f"planned raw I={effects['I'].mean():+.4f}; logit I={logit_effects['I'].mean():+.4f}")
print(f"sign-reversal example: raw I={reversal_raw:+.4f}; logit I={reversal_logit:+.4f}")
print(f"substitution-compatible={substitution}; full replacement={full_replacement}")

## 6. Participant-only and crossed sensitivity bootstraps

A participant draw is shared by every block and cell. A crossed bootstrap also samples block indices, and every selected block carries all four cells. The two procedures answer different repeated-sampling questions. Neither procedure turns bootstrap iterations into new participants or newly trained models.

In [ ]:
def boot_interaction(scores, block_draw, participant_draw):
    sampled = scores[block_draw][..., participant_draw]
    cells = sampled.mean(axis=-1, dtype=np.float64)
    return factorial_contrasts(cells)["I"].mean()

B = 800
boot_rng = np.random.Generator(np.random.PCG64(1701))
participant_only = np.empty(B)
crossed = np.empty(B)
fixed_blocks = np.arange(R)
for b in range(B):
    participant_draw = boot_rng.integers(0, P, size=P)
    participant_only[b] = boot_interaction(gfc, fixed_blocks, participant_draw)
    block_draw = boot_rng.integers(0, R, size=R)
    crossed[b] = boot_interaction(gfc, block_draw, participant_draw)

participant_ci = np.quantile(participant_only, [0.025, 0.975])
crossed_ci = np.quantile(crossed, [0.025, 0.975])
assert participant_only.shape == crossed.shape == (B,)
print("participant-only interval:", participant_ci)
print("crossed interval:         ", crossed_ci)

fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
ax.hist(crossed, bins=30, color="#f59e0b", edgecolor="white")
ax.axvline(0, color="#991b1b", linestyle=":")
ax.set(xlabel="mean block interaction", ylabel="bootstrap count",
       title="Crossed resampling preserves complete four-cell blocks")
plt.show()

## 7. Validate the 32-run registry before analysis

The registry must contain every block and cell exactly once, using the canonical labels `low`, `high`, `frozen_random`, and `resampled_anchor`. Each row records the count derived from its manifest, its optimization seed, its unique final-checkpoint identity, and every frozen stream version. The validator also checks strict low-within-high nesting and identical frozen anchors for shared sequences. Public summaries pass an exact recursive schema that rejects identifiers, private paths, strings, non-finite numbers, and unknown fields.

In [ ]:
SUPPORTS = ("low", "high")
POLICIES = ("frozen_random", "resampled_anchor")
FINAL_STEP = 64_000
FROZEN_FIELDS = {
    "exposure": 8_192_000,
    "anchor_spacing": 8,
    "window_seed_version": "window-v1",
    "temporal_stream_version": "temporal-v1",
    "spatial_stream_version": "spatial-v1",
    "mask_stream_version": "mask-v1",
    "protocol_revision": "hierarchy-v1",
    "final_step": FINAL_STEP,
}

def stable_digest(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def frozen_anchor(block, sequence_id):
    return 8 * ((37 * block + 11 * sequence_id + 5) % 9)

pool_members = {
    r: {"low": set(range(25)), "high": set(range(2500))}
    for r in range(R)
}
anchor_manifests = {
    r: {
        support: {seq: frozen_anchor(r, seq) for seq in pool_members[r][support]}
        for support in SUPPORTS
    }
    for r in range(R)
}

def manifest_payload(block, support, pools, anchors_by_support):
    sequence_ids = sorted(pools[block][support])
    return {
        "block": block,
        "support": support,
        "sequence_ids": sequence_ids,
        "frozen_anchors": [anchors_by_support[block][support][seq] for seq in sequence_ids],
        "anchor_spacing": FROZEN_FIELDS["anchor_spacing"],
    }

registry = []
for block, support, policy in product(range(R), SUPPORTS, POLICIES):
    model_label = f"r{block:02d}-{support}-{policy}"
    manifest_digest = stable_digest(manifest_payload(block, support, pool_members, anchor_manifests))
    registry.append({
        "block": block,
        "sequence_support": support,
        "window_policy": policy,
        "model_label": model_label,
        "manifest_digest": manifest_digest,
        "unique_sequences": len(pool_members[block][support]),
        "optimization_seed": 10_000 + block,
        "final_checkpoint_identity": f"{model_label}:step-{FINAL_STEP}",
        **FROZEN_FIELDS,
    })

REGISTRY_FIELDS = {
    "block", "sequence_support", "window_policy", "model_label",
    "manifest_digest", "unique_sequences", "optimization_seed",
    "final_checkpoint_identity", *FROZEN_FIELDS,
}

def validate_registry(rows, pools, anchors_by_support):
    if any(set(row) != REGISTRY_FIELDS for row in rows):
        raise ValueError("registry row schema changed")
    keys = [(row["block"], row["sequence_support"], row["window_policy"]) for row in rows]
    expected = set(product(range(R), SUPPORTS, POLICIES))
    if len(rows) != len(expected) or len(set(keys)) != len(keys) or set(keys) != expected:
        raise ValueError("registry must contain every canonical cell exactly once")
    for field, expected_value in FROZEN_FIELDS.items():
        if {row[field] for row in rows} != {expected_value}:
            raise ValueError(f"frozen field drift: {field}")
    labels = [row["model_label"] for row in rows]
    checkpoints = [row["final_checkpoint_identity"] for row in rows]
    if len(set(labels)) != len(labels) or len(set(checkpoints)) != len(checkpoints):
        raise ValueError("model and checkpoint identities must be unique")
    for row in rows:
        expected_checkpoint = f"{row['model_label']}:step-{FINAL_STEP}"
        if row["final_checkpoint_identity"] != expected_checkpoint:
            raise ValueError("checkpoint identity does not match the final step")
    for block in range(R):
        block_rows = [row for row in rows if row["block"] == block]
        if {row["optimization_seed"] for row in block_rows} != {10_000 + block}:
            raise ValueError("optimization seed is not shared within a block")
        if not pools[block]["low"] < pools[block]["high"]:
            raise ValueError("low support must be a strict subset of high support")
        for seq in pools[block]["low"]:
            if anchors_by_support[block]["low"].get(seq) != anchors_by_support[block]["high"].get(seq):
                raise ValueError("shared sequences changed their frozen anchor")
        for support in SUPPORTS:
            support_rows = [row for row in block_rows if row["sequence_support"] == support]
            expected_count = len(pools[block][support])
            expected_digest = stable_digest(manifest_payload(block, support, pools, anchors_by_support))
            if {row["unique_sequences"] for row in support_rows} != {expected_count}:
                raise ValueError("sequence count does not match its manifest")
            if {row["manifest_digest"] for row in support_rows} != {expected_digest}:
                raise ValueError("policies do not share the verified manifest")
    return True

assert validate_registry(registry, pool_members, anchor_manifests)

def must_reject_registry(rows, pools=pool_members, anchors_by_support=anchor_manifests):
    try:
        validate_registry(rows, pools, anchors_by_support)
    except (KeyError, TypeError, ValueError):
        return
    raise AssertionError("invalid registry was accepted")

stream_drift = [dict(row) for row in registry]
stream_drift[0]["mask_stream_version"] = "mask-v2"
must_reject_registry(stream_drift)
wrong_count = [dict(row) for row in registry]
wrong_count[0]["unique_sequences"] += 1
must_reject_registry(wrong_count)
changed_anchors = {r: {s: dict(anchor_manifests[r][s]) for s in SUPPORTS} for r in range(R)}
changed_anchors[0]["low"][0] += 8
changed_anchor_rows = [dict(row) for row in registry]
changed_digest = stable_digest(manifest_payload(0, "low", pool_members, changed_anchors))
for row in changed_anchor_rows:
    if row["block"] == 0 and row["sequence_support"] == "low":
        row["manifest_digest"] = changed_digest
must_reject_registry(changed_anchor_rows, anchors_by_support=changed_anchors)

PUBLIC_FIELDS = {"cell_means", "interaction", "complete_blocks"}
FORBIDDEN_KEY_PARTS = ("participant", "subject", "embedding", "path", "filename")
FORBIDDEN_STRING_PARTS = ("/private/", "/users/", ".mp4", ".npy", ".npz", ".pt")

def finite_number(value):
    numeric = isinstance(value, (int, float, np.integer, np.floating))
    return numeric and not isinstance(value, (bool, np.bool_)) and math.isfinite(float(value))

def privacy_scan(value, location="$"):
    if isinstance(value, dict):
        for key, child in value.items():
            if not isinstance(key, str):
                raise TypeError(f"non-string key at {location}")
            lowered = key.lower()
            if any(part in lowered for part in FORBIDDEN_KEY_PARTS):
                raise ValueError(f"private key at {location}.{key}")
            privacy_scan(child, f"{location}.{key}")
        return
    if isinstance(value, list):
        for index, child in enumerate(value):
            privacy_scan(child, f"{location}[{index}]")
        return
    if isinstance(value, str):
        lowered = value.lower()
        if any(part in lowered for part in FORBIDDEN_STRING_PARTS):
            raise ValueError(f"private path or artifact reference at {location}")
        raise TypeError(f"strings are not allowed in the public schema at {location}")
    if not finite_number(value):
        raise TypeError(f"unsupported or non-finite value at {location}")

def validate_public_summary(summary):
    if not isinstance(summary, dict) or set(summary) != PUBLIC_FIELDS:
        raise ValueError("public summary fields changed")
    privacy_scan(summary)
    means = summary["cell_means"]
    if not (isinstance(means, list) and len(means) == 2
            and all(isinstance(row, list) and len(row) == 2 for row in means)):
        raise ValueError("cell_means must be a two-by-two list")
    if not all(0.0 <= float(value) <= 1.0 for row in means for value in row):
        raise ValueError("cell means must be bounded scores")
    if not finite_number(summary["interaction"]):
        raise TypeError("interaction must be finite and numeric")
    blocks = summary["complete_blocks"]
    if not isinstance(blocks, int) or isinstance(blocks, bool) or blocks != R:
        raise ValueError("complete block count does not match the design")
    return True

public_summary = {
    "cell_means": Y.mean(axis=0).round(6).tolist(),
    "interaction": float(effects["I"].mean()),
    "complete_blocks": R,
}
assert validate_public_summary(public_summary)
path_injection = {**public_summary, "cell_means": [row[:] for row in public_summary["cell_means"]]}
path_injection["cell_means"][0][0] = "/private/healthgait/p001.mp4"
try:
    validate_public_summary(path_injection)
except (TypeError, ValueError):
    pass
else:
    raise AssertionError("private path injection was accepted")
print(f"validated {len(registry)} registry rows; public keys={list(public_summary)}")

## Exercises and takeaways

1. Lower the exposure in the support calculation. **Check:** both expectations fall, and the resampled capacity can remain mostly unrealized.
2. Resample 32 model rows independently instead of eight blocks. **Check:** the code can still return a number, but it no longer estimates the paired interaction.
3. Change the reversal example from `[0.90, 0.95]` to `[0.60, 0.65]`. **Check:** the raw and logit interactions no longer reverse.
4. Remove one registry row. **Check:** the exact cell-set assertion fails before analysis.

**Takeaways:** fixed exposure does not imply equal support; temporal support is an occupancy problem; four models form one paired block; the primary uncertainty uses eight block interactions; additive interactions depend on scale; equivalence needs its own margin and interval; and every sensitivity analysis must preserve blocks and participants.

## Continue learning

[Previous notebook: 16](16_reproducible_scientific_evaluators.ipynb) | [Lecture](../lectures/17_hierarchical_support_and_factorial_inference.md) | [Curriculum](../README.md)